In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openhexa.sdk import workspace, current_run
from openhexa.toolbox.dhis2 import DHIS2
from openhexa.toolbox.dhis2.dataframe import get_organisation_unit_groups

In [ ]:
# Parameters
# YEAR = 2026

In [ ]:
# Connect to DHIS2 SNIS
connection = workspace.dhis2_connection('drc-snis')
dhis2_source = DHIS2(connection=connection, cache_dir =None) 

### ORG unit groups download

- Download and format the file (expected columns for R scripts)

In [ ]:
# save (make sure we're saving the data for the corresponding year)
if int(YEAR) == datetime.now().year:
    
    # Retrieve org unit groups 
    groups = get_organisation_unit_groups(dhis2_source)
    groups_pd = groups.to_pandas()

    # check 1
    print(groups_pd[groups_pd["name"].str.contains("Service", case=False, na=False)])

    # and format
    groups_ren = groups_pd.rename(columns={"id":"oug_uid", "name":"oug_name", "organisation_units" : "org_units"})
    groups_ren["oug_shortname"] = groups_ren["oug_name"]
    groups_ren["org_units"] = groups_ren["org_units"].apply(lambda x: ";".join(x))
    groups_ren.loc[groups_ren["oug_uid"] == "LC3ogyAbDmI", "oug_shortname"] = "A - Services de Base"
    groups_ren.loc[groups_ren["oug_uid"] == "jo54HgxzRVL", "oug_shortname"] = "B - Services Secondaires"
    groups_ren.loc[groups_ren["oug_uid"] == "RZYPaa6gxq8", "oug_shortname"] = "D - Service Hôpital"

    # check 2
    print(groups_ren[groups_ren["oug_shortname"].str.contains("Service", case=False, na=False)])

    groups_ren.to_csv(f"/home/jovyan/workspace/pnlp-tdb-pipeline/data/metadata/organisation_unit_groups_{YEAR}.csv", index=False)
    groups_ren.to_csv("/home/jovyan/workspace/pnlp-tdb-pipeline/data/metadata/organisation_unit_groups.csv", index=False)
else:
    # Use the corresponding YEAR file.
    prev_year_oug = pd.read_csv(f"/home/jovyan/workspace/pnlp-tdb-pipeline/data/metadata/organisation_unit_groups_{YEAR}.csv")
    prev_year_oug.to_csv("/home/jovyan/workspace/pnlp-tdb-pipeline/data/metadata/organisation_unit_groups.csv", index=False)

### ORG units (pyramid)

-Load pyramid from DHIS2 SNIS Extracts pipeline data

In [ ]:
# save (make sure we're saving the data for the corresponding year)
if int(YEAR) == datetime.now().year:
    
    # org_units_meta = pd.read_parquet("/home/jovyan/workspace/pipelines/dhis2_snis_extract/data/raw/pyramid/snis_pyramid.parquet")
    # Loading pyramid downloaded from the shared dataset 'snis-extracts' (see pipeline: drc-pnlp-tdb)
    org_units_meta = pd.read_parquet("/home/jovyan/workspace/pnlp-tdb-pipeline/data/snis_pyramid/snis_pyramid.parquet")
    org_units_meta = org_units_meta.drop(columns=["shortName", "openingDate", "parent", "closedDate"])
    org_units_meta.rename(columns={"id" : "ou_uid", "name" : "ou_name", "level" : "ou_level"}, inplace=True)
    print(org_units_meta.head(2))
    print(f"OU at level 5: {org_units_meta[org_units_meta.ou_level == 5].shape[0]}")
    
    org_units_meta.to_csv(f"/home/jovyan/workspace/pnlp-tdb-pipeline/data/metadata/organisation_units_{YEAR}.csv", index=False)
    org_units_meta.to_csv("/home/jovyan/workspace/pnlp-tdb-pipeline/data/metadata/organisation_units.csv", index=False)
else:    
    prev_year_ous = pd.read_csv(f"/home/jovyan/workspace/pnlp-tdb-pipeline/data/metadata/organisation_units_{YEAR}.csv")
    prev_year_ous.to_csv("/home/jovyan/workspace/pnlp-tdb-pipeline/data/metadata/organisation_units.csv", index=False) # use the corresponding year file.